<a href="https://colab.research.google.com/github/K-Musty/oil-spill-detection/blob/main/notebook/oil_spill_detection-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OIL SPILL DETECTION

#### 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### 2. Install packages

In [ ]:
!pip install -q segmentation-models-pytorch albumentations onnx onnxruntime tqdm

#### 3. Clone repo

In [ ]:
!git clone https://github.com/K-Musty/oil-spill-detection.git
%cd oil-spill-detection

#### 4. Copy dataset and training scripts

In [ ]:
!cp src/dataset.py .
!cp src/train.py .
!cp src/train-deeplab.py .

#### 5. Create data directory and copy from Drive

In [ ]:
!mkdir -p data/processed
!cp /content/drive/MyDrive/oil-spill-detection/processed/train.npz data/processed/
!cp /content/drive/MyDrive/oil-spill-detection/processed/val.npz data/processed/
!cp /content/drive/MyDrive/oil-spill-detection/processed/test.npz data/processed/

#### 6. Verify data

In [ ]:
import numpy as np
import os

def check_data(name):
    data = np.load(f'data/processed/{name}.npz')
    print(f"{name}: images {data['images'].shape}, masks {data['masks'].shape}")

check_data('train')
check_data('val')
check_data('test')

In [ ]:
!python train.py --epochs 1 --batch_size 4

🚀 Using device: cuda
📊 DataLoaders created:
   Train: 955 samples
   Val:   205 samples
   Test:  205 samples
📊 Class distribution: Sea=0.9359, Oil=0.0641
📊 Class weights: [0.12824128568172455, 1.8717586994171143]
config.json: 100% 156/156 [00:00<00:00, 337kB/s]

model.safetensors: downloading bytes:  81% 70.3M/87.3M [00:00<00:00, 105MB/s, 4.95MB/s  ] 
model.safetensors: downloading bytes: 100% 80.6M/80.6M [00:00<00:00, 82.9MB/s, 7.77MB/s  ]
model.safetensors: reconstructing file: 100% 87.3M/87.3M [00:00<00:00, 89.7MB/s, 8.52MB/s  ]
📊 Model: unet with resnet34 backbone
📊 Parameters: 24,436,514

🔁 Epoch 1/1
📈 Train Loss: 0.7165 | Val IoU: 0.1876 | Val Dice: 0.3063
✅ Best model saved to checkpoints/unet_best_model.pt (IoU = 0.1876)

🏆 Best IoU: 0.1876 at epoch 1
                                               
✅ Test IoU: 0.2217
✅ Test Dice: 0.3430

✅ Training complete! Checkpoints and metrics saved to: checkpoints


#### 7. Train U-Net

In [ ]:
!python src/train.py --epochs 50 --batch_size 8

🚀 Using device: cuda
📊 DataLoaders created:
   Train: 955 samples
   Val:   205 samples
   Test:  205 samples
📊 Class distribution: Sea=0.9359, Oil=0.0641
📊 Class weights: [0.12824128568172455, 1.8717586994171143]
📊 Model: unet with resnet34 backbone
📊 Parameters: 24,436,514

🔁 Epoch 1/50
📈 Train Loss: 0.7417 | Val IoU: 0.1896 | Val Dice: 0.3092
✅ Best model saved to checkpoints/unet_best_model.pt (IoU = 0.1896)

🔁 Epoch 2/50
📈 Train Loss: 0.6258 | Val IoU: 0.1912 | Val Dice: 0.3134
✅ Best model saved to checkpoints/unet_best_model.pt (IoU = 0.1912)

🔁 Epoch 3/50
📈 Train Loss: 0.5866 | Val IoU: 0.2962 | Val Dice: 0.4492
✅ Best model saved to checkpoints/unet_best_model.pt (IoU = 0.2962)

🔁 Epoch 4/50
📈 Train Loss: 0.6004 | Val IoU: 0.1591 | Val Dice: 0.2682

🔁 Epoch 5/50
📈 Train Loss: 0.5787 | Val IoU: 0.3453 | Val Dice: 0.5056
✅ Best model saved to checkpoints/unet_best_model.pt (IoU = 0.3453)

🔁 Epoch 6/50
📈 Train Loss: 0.5514 | Val IoU: 0.3239 | Val Dice: 0.4829

🔁 Epoch 7/50
📈 Trai

Preserve the original 512 DeepLab checkpoint before rerunning at 256

In [ ]:
# Preserve the original 512 DeepLab checkpoint before rerunning at 256
import os, shutil
src = '/content/drive/MyDrive/oil-spill-detection/models/deeplabv3/deeplabv3_best_model.pt'
dst = '/content/drive/MyDrive/oil-spill-detection/models/deeplabv3/deeplabv3_512_best_model.pt'
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst)
    print(f"✅ Preserved 512 run → {dst}")
elif os.path.exists(dst):
    print(f"✅ 512 run already preserved at {dst}")
else:
    print("⚠️ 512 checkpoint not found in Drive — check the path")

#### 8. Train DeepLabV3

In [17]:
# !python src/train-deeplab.py --epochs 50 --batch_size 4 --lr 5e-5 --encoder resnet34


# If 512 was hardcoded in the pre-flight grep, uncomment and run ONCE to patch:
!cp src/train-deeplab.py src/train-deeplab.py.bak
!sed -i 's/512/256/g' src/train-deeplab.py
!grep -n "256\|512" src/train-deeplab.py | head

# Then rerun with U-Net's hyperparameters at 256
!python src/train-deeplab.py --epochs 50 --batch_size 8 --lr 1e-4 --encoder resnet34

3:# Standalone DeepLabV3 training with 256×256 input.
49:def get_transforms_256(train=True, image_size=256):
51:    Transform pipeline for 256×256 images.
72:def get_dataloaders_256(data_dir=None, batch_size=4, num_workers=2):
87:    train_dataset = OilSpillDataset(str(train_path), transform=get_transforms_256(train=True, image_size=256))
88:    val_dataset   = OilSpillDataset(str(val_path),   transform=get_transforms_256(train=False, image_size=256))
89:    test_dataset  = OilSpillDataset(str(test_path),  transform=get_transforms_256(train=False, image_size=256))
98:    print(f"📊 DataLoaders (256×256):")
186:    train_loader, val_loader, test_loader = get_dataloaders_256(
249:            f.write(f"Image size: 256\n")
🚀 Using device: cuda
📊 DataLoaders (256×256):
   Train: 955 samples
   Val:   205 samples
   Test:  205 samples
📊 Class distribution: Sea=0.9359, Oil=0.0641
📊 Class weights: [0.12824128568172455, 1.8717586994171143]
📊 Model: DeepLabV3 with resnet34 backbone
📊 Parameters: 

In [18]:
!cp checkpoints/deeplabv3_best_model.pt checkpoints/deeplabv3_256_best_model.pt
!ls -lh checkpoints/deeplabv3*.pt

-rw-r--r-- 1 root root 100M Oct 18 13:04 checkpoints/deeplabv3_256_best_model.pt
-rw-r--r-- 1 root root 100M Oct 18 13:02 checkpoints/deeplabv3_best_model.pt


In [19]:
# Create backup folder in Drive
backup_dir = '/content/drive/MyDrive/oil-spill-detection/models/deeplabv3/'
os.makedirs(backup_dir, exist_ok=True)

# Copy model checkpoint
checkpoint_path = 'checkpoints/deeplabv3_256_best_model.pt'
if os.path.exists(checkpoint_path):
    shutil.copy(checkpoint_path, backup_dir + 'deeplabv3_256_best_model.pt')
    print(f"✅ Copied: {checkpoint_path} → {backup_dir}")

# Copy metrics if they exist
metrics_path = 'checkpoints/deeplabv3_metrics.txt'
if os.path.exists(metrics_path):
    shutil.copy(metrics_path, backup_dir + 'deeplabv3_metrics.txt')
    print(f"✅ Copied: {metrics_path} → {backup_dir}")

# Verify
print("\n📊 Files in Drive:")
!ls -lh /content/drive/MyDrive/oil-spill-detection/models/deeplabv3/

✅ Copied: checkpoints/deeplabv3_256_best_model.pt → /content/drive/MyDrive/oil-spill-detection/models/deeplabv3/
✅ Copied: checkpoints/deeplabv3_metrics.txt → /content/drive/MyDrive/oil-spill-detection/models/deeplabv3/

📊 Files in Drive:
total 299M
-rw------- 1 root root 100M Oct 18 13:04 deeplabv3_256_best_model.pt
-rw------- 1 root root 100M Oct 18 12:41 deeplabv3_512_best_model.pt
-rw------- 1 root root 100M Sep  7 16:50 deeplabv3_best_model.pt
-rw------- 1 root root  165 Oct 18 13:04 deeplabv3_metrics.txt


#### 9. Train FPN

In [20]:
# !python src/train.py --model fpn --epochs 50 --batch_size 8 --lr 1e-4
!cp src/train.py src/train.py.bak
!sed -i 's/patience=10/patience=20/g' src/train.py
!grep -n "patience" src/train.py
!python src/train.py --model fpn --epochs 50 --batch_size 8 --lr 1e-4

179:        optimizer, mode='max', factor=0.5, patience=5
188:    patience_counter = 0
207:            patience_counter = 0
212:            patience_counter += 1
213:            if patience_counter >= args.patience:
241:        f.write(f"Patience: {args.patience}\n")
256:    parser.add_argument('--patience', type=int, default=10, help='Early stopping patience')
🚀 Using device: cuda
📊 DataLoaders created:
   Train: 955 samples
   Val:   205 samples
   Test:  205 samples
📊 Class distribution: Sea=0.9359, Oil=0.0641
📊 Class weights: [0.12824128568172455, 1.8717586994171143]
📊 Model: fpn with resnet34 backbone
📊 Parameters: 23,155,522

🔁 Epoch 1/50
📈 Train Loss: 0.8370 | Val IoU: 0.2769 | Val Dice: 0.4223
✅ Best model saved to checkpoints/fpn_best_model.pt (IoU = 0.2769)

🔁 Epoch 2/50
📈 Train Loss: 0.6731 | Val IoU: 0.2114 | Val Dice: 0.3391

🔁 Epoch 3/50
📈 Train Loss: 0.6457 | Val IoU: 0.2458 | Val Dice: 0.3813

🔁 Epoch 4/50
📈 Train Loss: 0.5512 | Val IoU: 0.3071 | Val Dice: 0.4609
✅ Best

In [ ]:
import torch, numpy as np

# --- adapt these two lines to your actual model loading API ---
from src.train import build_model   # <-- check the actual function name from pre-flight
# -------------------------------------------------------------

device = 'cuda' if torch.cuda.is_available() else 'cpu'
nooil = np.load('data/processed/nooil.npz')
imgs = nooil['images']   # (N, 256, 256, 3), all-zero masks

def evaluate_checkpoint(ckpt, model_name, threshold=0.5):
    model = build_model(model_name, encoder='resnet34').to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    fp_pix = 0; tot_pix = 0; fp_img = 0
    with torch.no_grad():
        for i in range(0, len(imgs), 8):
            b = torch.tensor(imgs[i:i+8]).permute(0,3,1,2).float().to(device)
            # apply the SAME normalization as training — confirm from dataset.py grep
            preds = (torch.sigmoid(model(b)) > threshold).cpu().numpy()
            fp_pix += preds.sum()
            tot_pix += preds.size
            fp_img += (preds.reshape(len(preds), -1).sum(axis=1) > 0).sum()
    print(f"{model_name:10s} pixel FPR={fp_pix/tot_pix:.4f}  image FP rate={fp_img/len(imgs):.4f}")

# use the actual filenames from the pre-flight `ls checkpoints/` output
evaluate_checkpoint('checkpoints/unet_best_model.pt',      'unet')        # <-- confirm name
evaluate_checkpoint('checkpoints/fpn_best_model.pt',       'fpn')
evaluate_checkpoint('checkpoints/deeplabv3_256_best_model.pt', 'deeplabv3')

#### 10. Save checkpoint to Drive

In [21]:
!mkdir -p /content/drive/MyDrive/oil-spill-detection/checkpoints
!cp checkpoints/best_model.pt /content/drive/MyDrive/oil-spill-detection/checkpoints/

print("✅ Training complete! Model saved to Drive.")

cp: cannot stat 'checkpoints/best_model.pt': No such file or directory
✅ Training complete! Model saved to Drive.


In [22]:
import os
import shutil

# Create backup folder in Drive
backup_dir = '/content/drive/MyDrive/oil-spill-detection/models/fpn/'
os.makedirs(backup_dir, exist_ok=True)

# Copy model checkpoint
checkpoint_path = 'checkpoints/fpn_best_model.pt'
if os.path.exists(checkpoint_path):
    shutil.copy(checkpoint_path, backup_dir + 'fpn_best_model.pt')
    print(f"✅ Copied: {checkpoint_path} → {backup_dir}")

# Copy metrics
metrics_path = 'checkpoints/fpn_metrics.txt'
if os.path.exists(metrics_path):
    shutil.copy(metrics_path, backup_dir + 'fpn_metrics.txt')
    print(f"✅ Copied: {metrics_path} → {backup_dir}")

# Verify
print("\n📊 Files in Drive:")
!ls -lh /content/drive/MyDrive/oil-spill-detection/models/fpn/

✅ Copied: checkpoints/fpn_best_model.pt → /content/drive/MyDrive/oil-spill-detection/models/fpn/
✅ Copied: checkpoints/fpn_metrics.txt → /content/drive/MyDrive/oil-spill-detection/models/fpn/

📊 Files in Drive:
total 89M
-rw------- 1 root root 89M Oct 18 13:09 fpn_best_model.pt
-rw------- 1 root root 177 Oct 18 13:09 fpn_metrics.txt
